In [0]:
import requests
import pandas as pd
from pprint import pprint
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window



In [0]:
# ✅ 공공데이터포털 API Key
# (URL Decoding Key 사용 권장)
SERVICE_KEY_1 = "qz6MbARyTC9CL2GrNus/xLfgJolMh3LYaY+kT98k6wDcHDbTZ/vRAvw8JGmU9PnK25a7lM/ePpU1Dl7AsKAyXw=="

In [0]:
url = "http://apis.data.go.kr/B552584/MsrstnInfoInqireSvc/getMsrstnList"

params = {
    "serviceKey": SERVICE_KEY_1,
    "returnType": "json",
    "numOfRows": "100",
    "pageNo": "1",
    "addr": "서울"
}

response = requests.get(url, params=params)
response.status_code

200

In [0]:
data = response.json()
pprint(data)

{'response': {'body': {'items': [{'addr': '서울 중구 덕수궁길 15 시청서소문별관 3동',
                                  'dmX': '37.564639',
                                  'dmY': '126.975961',
                                  'item': 'SO2, CO, O3, NO2, PM10, PM2.5',
                                  'mangName': '도시대기',
                                  'stationName': '중구',
                                  'year': '1995'},
                                 {'addr': '서울 용산구 한강대로 405 (서울역 앞)',
                                  'dmX': '37.549389',
                                  'dmY': '126.971519',
                                  'item': 'SO2, CO, O3, NO2, PM10, PM2.5',
                                  'mangName': '도로변대기',
                                  'stationName': '한강대로',
                                  'year': '1996'},
                                 {'addr': '서울 종로구 종로35가길 19 종로5,6가 동 주민센터',
                                  'dmX': '37.572025',
                                  'dmY':

In [0]:
items = data["response"]["body"]["items"]

stations_pdf = pd.DataFrame(items)

stations_pdf = stations_pdf[
    ["stationName", "addr", "dmX", "dmY"]
]

stations_sdf = spark.createDataFrame(stations_pdf)
stations_sdf.display()

stationName,addr,dmX,dmY
중구,서울 중구 덕수궁길 15 시청서소문별관 3동,37.564639,126.975961
한강대로,서울 용산구 한강대로 405 (서울역 앞),37.549389,126.971519
종로구,"서울 종로구 종로35가길 19 종로5,6가 동 주민센터",37.572025,127.005028
청계천로,서울 중구 청계천로 184 (청계천4가사거리 남강빌딩 앞),37.56865,126.998083
종로,서울 종로구 종로 169 (종묘주차장 앞),37.570633,126.996783
용산구,서울특별시 용산구 이태원로 224-19 (한남동) 한남로 복합문화센터,37.532057,127.002371
광진구,서울특별시 광진구 광나루로 571 구의 아리수정수센터,37.544639,127.095706
성동구,서울 성동구 뚝섬로3길 18 성수1가1동주민센터,37.542036,127.049685
강변북로,서울 성동구 강변북로 257 한강사업본부 옆,37.539283,127.040943
중랑구,서울 중랑구 용마산로 369 건강가정지원센터,37.584953,127.094283


In [0]:

stations_sdf.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("hive_metastore.demo_airstatus_bronze.BRZ_seoul_stations")
